# AURA Guard — Fine-tune YOLO26n on VisDrone (aerial/drone-view detector)

This notebook fine-tunes a lightweight YOLO model on **VisDrone2019-DET**, a dataset of real
drone-captured footage (288 video clips + ~10k images shot from actual UAVs over 14 cities,
angles ranging from near-straight-down to oblique) — not ground-level photos. That's what makes
the result actually work for a security drone's overhead camera, unlike the stock COCO-pretrained
weights most YOLO tutorials use.

**Before you run this:**
- `Runtime -> Change runtime type -> T4 GPU` (the free tier is enough).
- Training ~60 epochs at 640px takes roughly 1.5-3 hours on a free T4. Lower `EPOCHS` below for a
  faster, lower-accuracy first pass to sanity-check the whole pipeline before committing to a long run.
- **License note:** VisDrone is released under **CC BY-NC-SA — non-commercial, share-alike only**.
  This notebook (and the model it produces) is meant for personal/prototype use of AURA Guard. If you
  ever plan to sell or distribute the app commercially, you'd need a different training dataset, or a
  commercial license for VisDrone, before shipping a VisDrone-trained model.
- **Output:** a `model.tflite` file, downloaded automatically at the end. Drop it into
  `app/src/main/assets/models/model.tflite` in the AuraGuard project (overwrite any existing file).
  The app auto-detects a 10-class VisDrone output (vs. the 80-class COCO default) purely from the
  model's output shape and switches its label mapping automatically — no code changes needed.

In [ ]:
# Sanity-check that a GPU runtime is actually attached — training on CPU here would take days, not hours.
!nvidia-smi

In [ ]:
!pip install -U ultralytics -q
import ultralytics
ultralytics.checks()

In [ ]:
# ---- Training config — tweak these, then just "Run all" ----
EPOCHS = 60      # raise to 100-150 for the best accuracy; lower to ~20 for a quick first pass
IMGSZ = 640      # training/inference resolution. VisDrone objects are small in frame — 960 or 1280
                 # can noticeably improve small-object accuracy, at the cost of much slower training
                 # and a larger/slower on-device model. 640 is the practical default for a phone app.
BATCH = 16       # lower this (e.g. 8, or 4) if you hit a CUDA out-of-memory error
PROJECT = "aura_guard_visdrone"

In [ ]:
# Start from a pretrained checkpoint (transfer learning) rather than random weights — this is what
# makes 60 epochs on a free GPU enough to get a usable model instead of needing days of training.
# YOLO26 (2025) is preferred: faster CPU inference and better small-object accuracy than YOLOv8n,
# which matters a lot for a phone app looking at small, distant, overhead objects. If your installed
# ultralytics version doesn't have YOLO26 weights yet, this falls back to YOLOv8n automatically —
# either way the rest of this notebook works unchanged.
from ultralytics import YOLO

BASE_WEIGHTS = "yolo26n.pt"
try:
    model = YOLO(BASE_WEIGHTS)
    print(f"Loaded base weights: {BASE_WEIGHTS}")
except Exception as e:
    print(f"Could not load {BASE_WEIGHTS} ({e}); falling back to yolov8n.pt")
    BASE_WEIGHTS = "yolov8n.pt"
    model = YOLO(BASE_WEIGHTS)
    print(f"Loaded base weights: {BASE_WEIGHTS}")

In [ ]:
# data="VisDrone.yaml" is a dataset config bundled with ultralytics — it downloads the actual
# VisDrone2019-DET images/labels (~2.3GB) automatically the first time this runs. 10 classes:
# pedestrian, people, bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor.
results = model.train(
    data="VisDrone.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name="train",
    patience=15,      # stop early if validation mAP hasn't improved in 15 epochs
    exist_ok=True,
)

In [ ]:
# Quick sanity check on held-out validation data before exporting — mAP50 in the 0.3-0.5 range is
# a reasonable outcome for a nano model at 640px on VisDrone's small/dense objects; this is a much
# harder dataset than COCO street-level photos, so don't expect COCO-level numbers.
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.3f}   mAP50: {metrics.box.map50:.3f}")

In [ ]:
# Export the best checkpoint from training to TFLite. half=True gives a float16 model — a good
# default balance of size/speed/accuracy for a phone without needing a representative-data int8
# calibration pass. TFLiteObjectDetector.kt reads the model's actual input size at load time, so
# any square IMGSZ you trained at will work in the app without further changes.
best_weights = f"{PROJECT}/train/weights/best.pt"
export_model = YOLO(best_weights)
tflite_path = export_model.export(format="tflite", imgsz=IMGSZ, half=True)
print("Exported:", tflite_path)

In [ ]:
import os
import shutil
from google.colab import files

final_name = "model.tflite"
shutil.copy(tflite_path, final_name)
print(f"Ready: {final_name} ({os.path.getsize(final_name) / 1e6:.1f} MB)")
files.download(final_name)

## Next steps

1. The cell above downloads `model.tflite` to your computer.
2. In the AuraGuard project, place it at `app/src/main/assets/models/model.tflite`, overwriting any
   file already there.
3. Rebuild the app. `TFLiteObjectDetector` inspects the model's output shape at load time — seeing
   10 classes, it automatically switches from the COCO label set to the VisDrone one
   (`VisDroneLabels.kt`). No app code or config changes needed.
4. On real drone footage, a change detected inside an armed zone will now come back labeled
   (PERSON, CAR, TRUCK, MOTORCYCLE, BICYCLE) whenever the model is confident about what caused it;
   otherwise it falls back to the generic "CHANGE DETECTED" marker exactly as before this model
   existed. Detection still isn't tracked across frames — this only classifies a change that the
   change-detection engine already found, per the app's design (see `AuraViewModel.kt`).